# Step 1: I Loaded the train data set

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

In [ ]:
Train_data = pd.read_excel(r"C:\Users\Anthony Bannerman\Documents\heart disease analysis\train.xlsx")

In [ ]:
Train_data.head()

In [ ]:
Train_data.info()

In [ ]:
Train_data.describe()

# Convert categorical column target variable to numeric using the map function

In [ ]:
df = Train_data.copy()

In [ ]:
#convert target to 0/1
df['Heart Disease'] = df['Heart Disease'].map({'Presence':1, 'Absence':0})

In [ ]:
df.head()

In [ ]:
# Check unique values in target
print(df['Heart Disease'].value_counts())

# Identify categorical and numeric features

In [ ]:
categorical_cols = ['Sex', 'Chest pain type', 'EKG results', 'Exercise angina', 'Slope of ST', 'Thallium']
numeric_cols = [col for col in df.columns if col not in categorical_cols + ['Age', 'id', 'Heart Disease']]

# One-hot encode categorical variables

In [ ]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Correlation Analysis

In [ ]:
plt.figure(figsize=(12,10))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', cbar=True)
plt.title('Correlation Matrix of Features')
plt.show()

In [ ]:
from scipy.stats import pearsonr

In [ ]:
corr = df.corr()['Heart Disease'].sort_values(ascending=False)
print(corr)

In [ ]:
raw_features = ['Thallium', 'Chest pain type', 'Exercise angina', 'Number of vessels fluro', 'ST depression','Max HR']
    


# Separate Features and Target

In [ ]:
X = df[raw_features]
y = df['Heart Disease']

In [ ]:
X.head()

In [ ]:
y.head()

# Train–Validation Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Training the model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

# Fit the model
rf_model.fit(X_train_scaled, y_train)

In [ ]:
# Predictions
y_pred = rf_model.predict(X_val_scaled)

# Evaluating the model

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
auc_score = roc_auc_score(y_val, y_pred)
print("AUC-ROC:", auc_score)

# Import the Test Data and Sample Submission

In [93]:
Test_data = pd.read_excel(r"C:\Users\Anthony Bannerman\Documents\heart disease analysis\test.xlsx")

In [96]:
Test_data.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3
1,630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3
2,630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7
3,630003,44,0,3,112,177,0,0,168,0,0.9,1,0,3
4,630004,43,1,1,138,267,0,0,163,0,1.8,2,0,7


In [97]:
Test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270000 entries, 0 to 269999
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       270000 non-null  int64  
 1   Age                      270000 non-null  int64  
 2   Sex                      270000 non-null  int64  
 3   Chest pain type          270000 non-null  int64  
 4   BP                       270000 non-null  int64  
 5   Cholesterol              270000 non-null  int64  
 6   FBS over 120             270000 non-null  int64  
 7   EKG results              270000 non-null  int64  
 8   Max HR                   270000 non-null  int64  
 9   Exercise angina          270000 non-null  int64  
 10  ST depression            270000 non-null  float64
 11  Slope of ST              270000 non-null  int64  
 12  Number of vessels fluro  270000 non-null  int64  
 13  Thallium                 270000 non-null  int64  
dtypes: f

In [107]:
submission_data = pd.read_excel(r"C:\Users\Anthony Bannerman\Documents\heart disease analysis\sample_submission.xlsx")

In [108]:
submission_data.head()

,id,Heart Disease
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0


# Select the features

In [109]:
raw_features = [
    'Thallium',
    'Chest pain type',
    'Exercise angina',
    'Number of vessels fluro',
    'ST depression',
    'Max HR'
]

X_test = Test_data[raw_features]


# Step 2: Predict probabilities (for AUC)

In [110]:
y_test_prob = rf_model.predict_proba(X_test)[:, 1]

In [113]:
Submission_data = pd.DataFrame({
    'id': Test_data['id'],
    'Heart Disease': y_test_prob
})

Submission_data["Heart Disease"] = y_test_prob

Submission_data.to_csv("submission.csv", index=False)

# What I discovered?

The model correctly ranks a randomly chosen patient with heart disease above a patient without disease 87% of the time.